In [ ]:
!pip install -q requests beautifulsoup4 pandas openpyxl
print(" Dependencies installed")

In [ ]:
import requests
import json
import os
import time
import random
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import List, Optional
import pandas as pd

# --- Constants ---
BASE_URL = "https://www.daraz.com.np"
AJAX_SEARCH_URL = f"{BASE_URL}/catalog/?ajax=true&q="

# Headers to mimic a real browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.daraz.com.np/',
    'X-Requested-With': 'XMLHttpRequest',
}

@dataclass
class Product:
    title: str
    price: float
    original_price: Optional[float] = None
    discount: Optional[str] = None
    rating: Optional[float] = None
    review_count: Optional[int] = 0
    image_url: Optional[str] = None
    product_url: Optional[str] = None
    brand: Optional[str] = None
    location: Optional[str] = None
    scraped_at: str = None

    def to_dict(self):
        return asdict(self)

print(" Setup complete - Using requests (no Playwright needed)")

In [ ]:
def scrape_daraz_api(keyword, max_pages=2):
    """
    Optimized Daraz Nepal scraper - fetches product data via API.
    """
    all_products = []
    session = requests.Session()
    session.headers.update(HEADERS)
    
    print(f"Searching: {keyword} | Pages: {max_pages}")
    print("-" * 50)
    
    # Get initial cookies
    try:
        session.get(BASE_URL, timeout=10)
        time.sleep(1)
    except:
        pass
    
    for page in range(1, max_pages + 1):
        try:
            url = f"{AJAX_SEARCH_URL}{keyword.replace(' ', '+')}&page={page}"
            response = session.get(url, timeout=30)
            
            if response.status_code != 200:
                print(f"Page {page}: HTTP {response.status_code}")
                continue
            
            items = response.json().get("mods", {}).get("listItems", [])
            if not items:
                print(f"Page {page}: No more results")
                break
            
            # Process items
            for item in items:
                try:
                    title = item.get("name", "").strip()
                    if not title:
                        continue
                    
                    price = float(item.get("price", 0))
                    original_price = item.get("originalPrice", "")
                    original_price = float(original_price) if original_price else None
                    
                    # Calculate discount
                    discount = item.get("discount", "")
                    if not discount and original_price and original_price > price:
                        discount = f"-{int((1 - price/original_price) * 100)}%"
                    
                    rating = item.get("ratingScore", "")
                    rating = float(rating) if rating else None
                    review_count = int(item.get("review", 0)) if item.get("review") else 0
                    
                    # Fix URLs
                    image_url = item.get("image", "")
                    if image_url.startswith("//"): image_url = "https:" + image_url
                    
                    product_url = item.get("productUrl", "")
                    if product_url.startswith("//"): product_url = "https:" + product_url
                    elif product_url.startswith("/"): product_url = BASE_URL + product_url
                    
                    all_products.append(Product(
                        title=title[:200],
                        price=price,
                        original_price=original_price,
                        discount=discount,
                        rating=rating,
                        review_count=review_count,
                        image_url=image_url,
                        product_url=product_url,
                        brand=item.get("brandName", ""),
                        location=item.get("location", ""),
                        scraped_at=datetime.now().isoformat()
                    ))
                except:
                    continue
            
            print(f"Page {page}: {len([p for p in all_products if p])} products")
            
            # Rate limiting
            if page < max_pages and items:
                time.sleep(random.uniform(2, 4))
                
        except requests.Timeout:
            print(f"Page {page}: Timeout")
        except Exception as e:
            print(f"Page {page}: Error - {str(e)[:50]}")
    
    print(f"\nTotal: {len(all_products)} products scraped")
    return all_products

print("Scraper ready!")

In [ ]:
import base64
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime
import io
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 10, 'figure.max_open_warning': 0})
plt.style.use('ggplot')

# ── STEP 1: User input ────────────────────────────────────────────────────────
search_keyword = input("Enter product to search: ").strip() or "fan"
pages = int(input("Number of pages to scrape (press Enter for 2): ").strip() or 2)

print(f"\n{'='*60}")
print(f"DARAZ NEPAL - {search_keyword.upper()} | Pages: {pages}")
print(f"{'='*60}")

# ── STEP 2: Scrape ────────────────────────────────────────────────────────────
products_data = scrape_daraz_api(search_keyword, max_pages=pages)

# ── HELPERS ───────────────────────────────────────────────────────────────────
def create_chart(df, chart_type='pie'):
    """Create either pie or bar chart with 4 key metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{search_keyword.title()} - {len(df)} Products', fontsize=14, fontweight='bold')
    
    # Color palette
    colors = plt.cm.Set3(np.linspace(0, 1, 10))
    
    # 1. Price Distribution
    if 'price' in df.columns and df['price'].notna().any():
        bins = [0, 1000, 5000, 10000, 25000, 50000, 100000, float('inf')]
        labels = ['<1K', '1K-5K', '5K-10K', '10K-25K', '25K-50K', '50K-100K', '>100K']
        dist = pd.cut(df['price'], bins=bins, labels=labels).value_counts()
        plot_chart(axes[0,0], dist, 'Price (Rs.)', chart_type, colors)
    
    # 2. Top Brands (excluding "No Brand")
    if 'brand' in df.columns:
        brands = df[df['brand'].str.lower() != 'no brand']['brand'].value_counts().head(8)
        if len(brands) > 0:
            plot_chart(axes[0,1], brands, 'Top Brands', chart_type, colors)
        else:
            axes[0,1].text(0.5, 0.5, 'No Brands Available', ha='center', transform=axes[0,1].transAxes)
            axes[0,1].set_title('Top Brands')
    
    # 3. Rating Distribution
    if 'rating' in df.columns and df['rating'].notna().any():
        bins = [0, 3, 3.5, 4, 4.5, 5]
        labels = ['<3.0', '3.0-3.5', '3.5-4.0', '4.0-4.5', '4.5-5.0']
        dist = pd.cut(df['rating'].dropna(), bins=bins, labels=labels).value_counts().sort_index()
        plot_chart(axes[1,0], dist, 'Ratings', chart_type, colors)
    
    # 4. Discount Distribution
    if 'discount' in df.columns:
        discounted = df[df['discount'].notna() & (df['discount'] > 0)]
        if len(discounted) > 0:
            bins = [0, 10, 20, 30, 50, 100]
            labels = ['<10%', '10-20%', '20-30%', '30-50%', '>50%']
            dist = pd.cut(discounted['discount'], bins=bins, labels=labels).value_counts()
            plot_chart(axes[1,1], dist, 'Discounts', chart_type, colors)
        else:
            axes[1,1].text(0.5, 0.5, 'No Discounts', ha='center', transform=axes[1,1].transAxes)
            axes[1,1].set_title('Discounts')
    
    plt.tight_layout()
    return fig

def plot_chart(ax, data, title, chart_type, colors):
    """Plot either pie or bar chart on given axis"""
    if chart_type == 'pie':
        wedges, texts, autotexts = ax.pie(
            data.values, labels=data.index, autopct='%1.1f%%',
            colors=colors[:len(data)], startangle=90, pctdistance=0.8
        )
        for t in texts: t.set_fontsize(9)
        for t in autotexts: t.set_fontsize(8)
    else:  # bar
        bars = ax.bar(range(len(data)), data.values, color=colors[:len(data)], edgecolor='black', linewidth=0.5)
        ax.set_xticks(range(len(data)))
        ax.set_xticklabels(data.index, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Count', fontsize=9)
        # Add value labels
        for bar, val in zip(bars, data.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), str(val),
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_title(title, fontsize=12, fontweight='bold')

def build_excel(df):
    """Create Excel file with multiple sheets"""
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Products', index=False)
        
        # Summary
        summary = {
            'Metric': ['Keyword', 'Pages', 'Products', 'Avg Price', 'Min Price', 'Max Price', 'Avg Rating', 'Reviews'],
            'Value': [
                search_keyword, pages, len(df),
                f"Rs.{df['price'].mean():,.0f}" if 'price' in df.columns else 'N/A',
                f"Rs.{df['price'].min():,.0f}" if 'price' in df.columns else 'N/A',
                f"Rs.{df['price'].max():,.0f}" if 'price' in df.columns else 'N/A',
                f"{df['rating'].mean():.1f}/5" if 'rating' in df.columns else 'N/A',
                df['review_count'].sum() if 'review_count' in df.columns else 0
            ]
        }
        pd.DataFrame(summary).to_excel(writer, sheet_name='Summary', index=False)
        
        # Top Brands
        if 'brand' in df.columns:
            brands = df[df['brand'].str.lower() != 'no brand']['brand'].value_counts().head(15)
            pd.DataFrame({'Brand': brands.index, 'Count': brands.values}).to_excel(writer, sheet_name='Top Brands', index=False)
    return buf.getvalue()

# ── STEP 3: Process & Display ─────────────────────────────────────────────────
if products_data and len(products_data) > 0:
    df = pd.DataFrame([p.to_dict() for p in products_data])
    for col in ['price', 'discount', 'rating', 'review_count']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Quick summary
    print(f"\nProducts: {len(df)} | Avg Price: Rs.{df['price'].mean():,.0f} | Avg Rating: {df['rating'].mean():.1f}/5")
    if 'brand' in df.columns:
        top_brands = df[df['brand'].str.lower() != 'no brand']['brand'].value_counts().head(3)
        if not top_brands.empty:
            print(f"Top Brands: {', '.join([f'{b}({c})' for b,c in top_brands.items()])}")
    
    display(df[['title','price','rating','brand']].head(8))
    
    # ── STEP 4: Download Menu ─────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print("DOWNLOAD: 1=Excel  2=Pie  3=Bar  4=Pie+Bar  5=All  6=Skip")
    choice = input("Choice (1-6): ").strip()
    
    if choice in ['1','2','3','4','5']:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        kw = search_keyword.replace(' ', '_')
        btns = ''
        
        if choice in ['1','5']:  # Excel
            xl_bytes = build_excel(df)
            btns += f'<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{base64.b64encode(xl_bytes).decode()}" download="daraz_{kw}_{ts}.xlsx" style="display:inline-block;padding:10px 20px;background:#217346;color:white;text-decoration:none;border-radius:5px;margin:5px;font-weight:bold;">Download Excel ({len(xl_bytes)/1024:.0f}KB)</a>'
        
        if choice in ['2','4','5']:  # Pie
            fig = create_chart(df, 'pie')
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
            plt.close(fig)
            btns += f'<a href="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}" download="daraz_{kw}_pie_{ts}.png" style="display:inline-block;padding:10px 20px;background:#1565C0;color:white;text-decoration:none;border-radius:5px;margin:5px;font-weight:bold;">Download Pie Chart</a>'
        
        if choice in ['3','4','5']:  # Bar
            fig = create_chart(df, 'bar')
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
            plt.close(fig)
            btns += f'<a href="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}" download="daraz_{kw}_bar_{ts}.png" style="display:inline-block;padding:10px 20px;background:#E65100;color:white;text-decoration:none;border-radius:5px;margin:5px;font-weight:bold;">Download Bar Chart</a>'
        
        display(HTML(f'<div style="padding:15px;background:#f0f8ff;border:2px solid #4CAF50;border-radius:8px;margin:10px 0;"><h3 style="color:#2e7d32;">Ready!</h3>{btns}</div>'))
else:
    print("No products found.")